# पाठ 13 - एजेंट मेमोरी


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-4.1-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## एजेंट मेमोरी के प्रकार

एआई एजेंट विभिन्न प्रकार की मेमोरी का उपयोग कर सकते हैं, जिनमें से प्रत्येक का एक विशिष्ट उद्देश्य होता है:

### वर्किंग मेमोरी
बातचीत थ्रेड स्वयं — एक सत्र में आदान-प्रदान किए गए संदेश। एजेंट उसी थ्रेड में पहले के संदेशों का संदर्भ लेकर सामंजस्य बनाए रख सकता है। MAF में यह **`agent.create_session()`** के माध्यम से बनाया जाता है, जो एक `AgentSession` लौटाता है।

### शॉर्ट-टर्म मेमोरी
जानकारी जो कार्य या सत्र की अवधि तक बनी रहती है लेकिन स्थायी रूप से संग्रहित नहीं होती। उदाहरण के लिए, एजेंट एक बहु-चर्चा योजना वार्तालाप के दौरान तथ्यों को इकट्ठा कर सकता है और उनका उपयोग अंतिम यात्रा कार्यक्रम बनाने के लिए कर सकता है।

### लॉन्ग-टर्म मेमोरी
ऐसी पसंद और तथ्य जो **सत्रों के बीच** बनी रहती हैं। एक लौटने वाले उपयोगकर्ता को अपनी डाइट संबंधी प्रतिबंध या यात्रा शैली दोहरानी नहीं पड़नी चाहिए। लॉन्ग-टर्म मेमोरी आमतौर पर एक बाहरी संग्रह द्वारा समर्थित होती है — एक डेटाबेस, फ़ाइल, या वेक्टर इंडेक्स — और उपकरणों के माध्यम से एजेंट के समक्ष प्रस्तुत की जाती है।


## सत्रों के साथ कार्यकारी स्मृति

स्मृति का सबसे सरल रूप वार्तालाप सत्र है। जब आप एक ही सत्र (`agent.create_session()` के माध्यम से बनाया गया) को लगातार `agent.run()` कॉल में पास करते हैं, तो एजेंट उस वार्तालाप का पूरा इतिहास देख सकता है और पहले के विवरण याद रख सकता है।

आइए एक यात्रा एजेंट बनाएं और कार्यकारी स्मृति का प्रदर्शन करें।


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजेंट ने बजट को सही ढंग से याद किया क्योंकि दोनों संदेशों का एक ही सत्र साझा था। यह **कार्यशील स्मृति** है — यह केवल सत्र के जीवनकाल के लिए मौजूद होती है।

### एक नए थ्रेड के साथ क्या होता है?

यदि हम एक **नया** सत्र बनाते हैं, तो एजेंट को पिछली बातचीत की कोई याद नहीं होगी:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## दीर्घकालिक स्मृति पैटर्न

उपयोगकर्ता की प्राथमिकताओं को **सत्रों के पार** याद रखने के लिए, हमें एक दृढ़ भंडार की आवश्यकता होती है जो बातचीत धागे के बाहर रहता है। एजेंट इस भंडार तक पहुँचता है **उपकरणों** के माध्यम से — ऐसे फ़ंक्शंस जिन्हें वह जानकारी को सहेजने और पुनः प्राप्त करने के लिए कॉल कर सकता है।

नीचे हम एक सरल इन-मेमोरी प्राथमिकता भंडार लागू करते हैं (उत्पादन में आप इसे डेटाबेस या वेक्टर इंडेक्स के साथ बैक करेंगे) और इसे ऐसे उपकरणों के रूप में प्रस्तुत करते हैं जिनका एजेंट उपयोग कर सकता है।

### वास्तुकला
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### परिदृश्य 1 — पहली बार उपयोगकर्ता सालगिरह यात्रा बुक करता है

सारा पहली बार आती है। एजेंट को उपकरणों के माध्यम से उसकी प्राथमिकताओं को संग्रहित करना चाहिए और होटल की सिफारिश करने के लिए उनका उपयोग करना चाहिए।


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### परिदृश्य 2 — सारा कुछ सप्ताह बाद लौटती है

सारा एक **बिल्कुल नया थ्रेड** शुरू करती है (एक नया सेशन सिमुलेट करते हुए)। वर्किंग मेमोरी खाली है, लेकिन लॉन्ग-टर्म प्रेफरेंस स्टोर में अभी भी उसकी जानकारी है। एजेंट को इसे पुनः प्राप्त करना चाहिए और व्यक्तिगत सिफारिशों के लिए इसका उपयोग करना चाहिए।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

इस पाठ में आपने तीन प्रकार की एजेंट मेमोरी और उन्हें माइक्रोसॉफ्ट एजेंट फ्रेमवर्क के साथ कैसे लागू किया जाए, सीख लिया:

| मेमोरी प्रकार | MAF तंत्र | जीवनकाल |
|---|---|---|
| **कार्यशील** | `agent.create_session()` | एकल बातचीत |
| **अल्पकालिक** | एक थ्रेड के भीतर संचित संदर्भ | एकल कार्य / सत्र |
| **दीर्घकालिक** | बाहरी स्टोर, जिसे `@tool` फ़ंक्शंस के माध्यम से एक्सेस किया जाता है | सत्रों के पार |

### मुख्य निष्कर्ष
1. **`agent.create_session()`** कार्यशील मेमोरी प्रदान करता है — एजेंट सत्र के भीतर पूरी बातचीत का इतिहास देखता है।
2. **नए सत्र संदर्भ खो देते हैं** — बिना दीर्घकालिक मेमोरी के एजेंट पिछली बातचीत याद नहीं कर सकता।
3. **`@tool` फ़ंक्शंस** इस अंतर को पाटते हैं — ये एजेंट को स्थायी स्टोर से जानकारी सहेजने और पुनः प्राप्त करने देते हैं।
4. **व्यक्तिगतकरण समय के साथ बेहतर होता है** — जितनी अधिक प्राथमिकताएँ संग्रहीत होती हैं, एजेंट की सिफारिशें उतनी बेहतर होती हैं।

### वास्तविक दुनिया के अनुप्रयोग
- **ग्राहक सेवा**: ग्राहक इतिहास और प्राथमिकताओं को याद रखें
- **व्यक्तिगत सहायक**: दिनों या हफ्तों के दौरान संदर्भ बनाए रखें
- **स्वास्थ्य सेवा**: रोगी की जानकारी और प्राथमिकताओं को ट्रैक करें
- **ई-कॉमर्स**: इतिहास के आधार पर व्यक्तिगत खरीदारी

### अगले चरण
- इन-मेमोरी डिक्ट को एक डेटाबेस या वेक्टर स्टोर (जैसे Azure AI Search) से बदलें
- समय-संवेदनशील जानकारी के लिए मेमोरी समाप्ति जोड़ें
- साझा मेमोरी वाले मल्टी-एजेंट सिस्टम बनाएं
- ज्ञान-ग्राफ समर्थित मेमोरी के लिए Cognee नोटबुक का अन्वेषण करें


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
